# Iris classifier with a small neural network

A minimal multilayer perceptron (MLP) for the classic Iris dataset: 150 flowers, 4 measurements each, 3 species.

Network: **4 inputs → hidden layer(s) → 3 outputs** (softmax), ReLU activations, trained with Adam.

In [1]:
from sklearn.datasets import load_iris
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

## Settings

Use `(16,)` for one hidden layer or `(16, 8)` for two.

In [2]:
HIDDEN_LAYERS = (16, 8)   # two hidden layers: 16 and 8 neurons
SEED = 42

## Data

Hold out 30% for testing, stratified so each species is equally represented in both splits.

In [3]:
iris = load_iris()
X, y = iris.data, iris.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=SEED
)
print(f"Train: {X_train.shape[0]} samples, test: {X_test.shape[0]} samples")

Train: 105 samples, test: 45 samples


## Model

Inputs are standardized before the network, which helps gradient-based training converge.

In [4]:
model = make_pipeline(
    StandardScaler(),
    MLPClassifier(
        hidden_layer_sizes=HIDDEN_LAYERS,
        activation="relu",
        solver="adam",
        learning_rate_init=0.01,
        max_iter=1000,
        random_state=SEED,
    ),
)
model.fit(X_train, y_train)

Pipeline(steps=[('standardscaler', StandardScaler()),
                ('mlpclassifier',
                 MLPClassifier(hidden_layer_sizes=(16, 8),
                               learning_rate_init=0.01, max_iter=1000,
                               random_state=42))])

## Evaluation

In [5]:
y_pred = model.predict(X_test)
print(f"Hidden layers: {HIDDEN_LAYERS}")
print(f"Train accuracy: {model.score(X_train, y_train):.3f}")
print(f"Test accuracy:  {model.score(X_test, y_test):.3f}\n")
print("Confusion matrix (rows = true, cols = predicted):")
print(confusion_matrix(y_test, y_pred), "\n")
print(classification_report(y_test, y_pred, target_names=iris.target_names))

Hidden layers: (16, 8)
Train accuracy: 1.000
Test accuracy:  0.911

Confusion matrix (rows = true, cols = predicted):
[[15  0  0]
 [ 0 15  0]
 [ 0  4 11]] 

              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        15
  versicolor       0.79      1.00      0.88        15
   virginica       1.00      0.73      0.85        15

    accuracy                           0.91        45
   macro avg       0.93      0.91      0.91        45
weighted avg       0.93      0.91      0.91        45



Setosa is separated perfectly; the remaining errors are between versicolor and virginica, which overlap in feature space. Training accuracy above test accuracy suggests mild overfitting — try a smaller network or stronger L2 regularization (`alpha` in `MLPClassifier`).